# 📊 Análise — Indicadores de Produção Industrial

Notebook de análise exploratória sobre os KPIs gerados na camada Gold.

**Análises disponíveis:**
1. Resumo do pipeline — volume de registros por camada
2. OEE por equipamento — ranking de eficiência
3. Taxa de defeitos por turno — identificação do turno mais crítico
4. Performance por linha de produção — comparativo entre linhas

> 💡 Cada consulta possui uma visualização em gráfico de barras configurada via Databricks Visualizations.

## 1. Resumo do Pipeline

Verificação do volume de registros em cada camada da arquitetura Medallion.

**Esperado:**
- Bronze: 15 registros (dados brutos com anomalias)
- Silver: 12 registros (após limpeza de qualidade)
- Gold: 11 registros (agregados por linha + turno + equipamento)

In [0]:
%sql
SELECT 'BRONZE' AS camada, COUNT(*) AS total FROM bronze.production_events
UNION ALL
SELECT 'SILVER' AS camada, COUNT(*) AS total FROM silver.production_events
UNION ALL
SELECT 'GOLD'   AS camada, COUNT(*) AS total FROM gold.production_kpis;

## 2. OEE por Equipamento

Ranking de eficiência global dos equipamentos, agrupado por linha de produção.

**Como interpretar o OEE:**
- `≥ 85%` → Classe mundial
- `60% – 85%` → Médio, há espaço para melhoria
- `< 60%` → Crítico, requer ação imediata

> 📈 Visualização sugerida: Bar Chart | X: `equipment_id` | Y: `oee_medio` | Group by: `line_id`

In [0]:
%sql
SELECT
  equipment_id,
  line_id,
  ROUND(AVG(oee_pct), 2)          AS oee_medio,
  ROUND(AVG(availability_pct), 2) AS disponibilidade_media,
  ROUND(AVG(quality_pct), 2)      AS qualidade_media,
  SUM(total_production)           AS producao_total
FROM gold.production_kpis
GROUP BY equipment_id, line_id
ORDER BY oee_medio DESC;

## 3. Taxa de Defeitos por Turno

Identificação do turno com maior índice de não conformidade. Útil para direcionar ações de qualidade e treinamento.

> 📈 Visualização sugerida: Bar Chart | X: `shift` | Y: `taxa_defeito_media`

In [0]:
%sql
SELECT
  shift,
  ROUND(AVG(defect_rate_pct), 2) AS taxa_defeito_media,
  SUM(total_defects)             AS total_defeitos,
  SUM(total_production)          AS total_producao
FROM gold.production_kpis
GROUP BY shift
ORDER BY taxa_defeito_media DESC;

## 4. Performance por Linha de Produção

Comparativo consolidado entre as linhas LINE-A, LINE-B e LINE-C. Permite identificar qual linha exige maior atenção operacional.

> 📈 Visualização sugerida: Bar Chart | X: `line_id` | Y: `oee_medio`

In [0]:
%sql
SELECT
  line_id,
  ROUND(AVG(oee_pct), 2)       AS oee_medio,
  SUM(total_production)        AS producao_total,
  SUM(total_downtime_min)      AS downtime_total_min,
  SUM(total_defects)           AS defeitos_total
FROM gold.production_kpis
GROUP BY line_id
ORDER BY oee_medio DESC;